# Chapter 7 &mdash; Nondeterminism as Forking Tokens

**Concept 1 of the Chapter 7 decomposition:** *Nondeterminism as Forking Tokens, and as Guessing*

A token splits on an input; each copy pursues one guess; any copy reaching a final state wins.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Forking-Tokens/Concept-Forking-Tokens.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Picture a **token** sitting on the start state. When an input arrives and several
edges carry that symbol, the token **forks** &mdash; one copy down each edge. Copies with
nowhere to go simply **die**.

The string is accepted if **any** copy is sitting on a final state when the input runs
out. That is the "**angelic**" reading of nondeterminism: it is enough that *some*
guess works out.

The equivalent reading is **guessing**: the machine guesses the right path and is
never wrong when a right path exists.

## 2. Definitions

### An NFA that guesses where the pattern starts

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I        !! stay and wait -- guess 'not yet'
I : 1 -> A            !! guess 'the pattern starts HERE'
A : 0 | 1 -> B
B : 0 | 1 -> F
''')
print("Q0 (a SET) :", N["Q0"])
print("states     :", sorted(N["Q"]))

### Watch the token set fork, step by step

In [ ]:
def fork_trace(N, s):
    cur = Eclosure(N, N["Q0"])
    print("start  : %s" % sorted(cur))
    for ch in s:
        nxt = set()
        for q in cur:
            nxt |= step_nfa(N, q, ch)
        cur = Eclosure(N, nxt)
        print("on '%s' : %-28s %s" % (ch, sorted(cur),
              "<- a copy is on a final state" if cur & N["F"] else ""))
    return cur

## 3. Tests

The token set grows and shrinks as copies fork and die.

In [ ]:
end = fork_trace(N, '01100')      # third-last symbol is the middle 1
print("\naccepted?", bool(end & N["F"]), " accepts_nfa says", accepts_nfa(N, '01100'))
assert accepts_nfa(N, '01100')
assert not accepts_nfa(N, '1001')     # third-last here is a 0

Only **four** states, yet the language is the exponential one from Chapter 5.

In [ ]:
print("|Q| of this NFA :", len(N["Q"]))
spec = lambda s: len(s) >= 3 and s[-3] == '1'
from itertools import product
bad = [''.join(p) for k in range(10) for p in product('01', repeat=k)
       if accepts_nfa(N, ''.join(p)) != spec(''.join(p))]
print("mismatches with 'third-last symbol is 1' :", len(bad))
assert not bad

Copies that guessed wrong die quietly &mdash; they never cause rejection.

In [ ]:
print("after '11' the token set is", sorted(Eclosure(N, {q for p in Eclosure(N, N["Q0"])
      for q in step_nfa(N, p, '1')} )))
print("one copy is still waiting in I; others are committed to a guess.")
print("\naccepts '110'?", accepts_nfa(N, '110'), " -- the copy that guessed at")
print("position 1 reaches F, so the wrong guesses do not matter.")
assert accepts_nfa(N, '110')

## 4. Animation

Four states. Watch the token fork at every 1 and the wrong copies die off.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. What would "**demonic**" nondeterminism mean &mdash; accept only if *every* copy succeeds?
2. How many copies exist after reading `1111`?
3. Write the NFA for "the **fourth**-last symbol is a 1". How many states?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter7/Concept-Forking-Tokens')